# OmniVoice Studio — Kaggle Local SSD + AI-native Server

The execution workspace remains **Kaggle local SSD only**:

```text
/kaggle/working/OmniVoiceStudio
├── projects/
├── voices/
├── project-queue.json
├── jobs.json
└── hardware-quality.json
```

The unified server exposes Gradio `/ui`, REST `/api/v1`, SSE job progress, and MCP `/mcp` from one process. Google Drive/rclone persistence is still intentionally out of scope.


In [ ]:
# Kaggle Internet must be enabled for GitHub/Hugging Face downloads.
!pip install -q --upgrade "git+https://github.com/binhminhanh1235/OmniVoice.git@feat/stable-tunnel-v1"


In [ ]:
from pathlib import Path
import shutil
import torch

from omnivoice.runtime_workspace import detect_runtime_workspace, ensure_runtime_workspace
from omnivoice.hardware_quality import detect_hardware

runtime = ensure_runtime_workspace(detect_runtime_workspace())
hardware = detect_hardware()

print("Runtime:", runtime.summary())
print("Hardware:", hardware.summary())
print("Workspace:", runtime.root)
print("Input source:", runtime.input_root)
usage = shutil.disk_usage(Path("/kaggle/working"))
print(f"Local disk free: {usage.free / 1024**3:.1f} GB")

if runtime.environment != "kaggle":
    raise RuntimeError(f"Expected Kaggle runtime, detected: {runtime.environment}")
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU accelerator in Kaggle Notebook settings.")


## Optional but recommended: stable hostname

Create one remotely-managed Cloudflare Tunnel once and map your hostname, for example `omnivoice.example.com`, to `http://localhost:8000`. Put the tunnel token in **Kaggle Secrets** using the name `CLOUDFLARE_TUNNEL_TOKEN`.

The token is never written into this notebook or repository. Each new Kaggle session reconnects the same named tunnel, so ChatGPT / Claude Code / Antigravity can keep one MCP URL.


In [ ]:
# Set this to your stable hostname configured in Cloudflare.
PUBLIC_URL = "https://omnivoice.example.com"
USE_STABLE_TUNNEL = True

import os

if USE_STABLE_TUNNEL:
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("CLOUDFLARE_TUNNEL_TOKEN")
    if not token:
        raise RuntimeError("Kaggle Secret CLOUDFLARE_TUNNEL_TOKEN is missing.")
    os.environ["CLOUDFLARE_TUNNEL_TOKEN"] = token
    os.environ["OMNIVOICE_PUBLIC_URL"] = PUBLIC_URL
    del token
    print("Stable public URL configured:", PUBLIC_URL)


In [ ]:
# Install the official Cloudflare Linux amd64 connector into ephemeral /kaggle/working.
# Studio itself never downloads network executables.
if USE_STABLE_TUNNEL:
    !wget -q -O /kaggle/working/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod 700 /kaggle/working/cloudflared
    !/kaggle/working/cloudflared --version


## Launch OmniVoice Studio

Recommended workflow inside `/ui`: Text Doctor → Voice Doctor → Project Studio → Project Queue → Hardware & Quality.

With the stable tunnel enabled, the same hostname exposes:

```text
/ui       human Web UI
/api/v1   REST API
/mcp      ChatGPT / Claude / Antigravity
/health   health check
```


In [ ]:
if USE_STABLE_TUNNEL:
    !omnivoice-studio serve \
      --model k2-fsa/OmniVoice \
      --workspace /kaggle/working/OmniVoiceStudio \
      --asr-model openai/whisper-small.en \
      --asr-device cpu \
      --host 0.0.0.0 \
      --port 8000 \
      --tunnel \
      --cloudflared /kaggle/working/cloudflared \
      --public-url $OMNIVOICE_PUBLIC_URL
else:
    print("Stable tunnel disabled. Using legacy temporary Gradio share URL.")
    !omnivoice-project-studio \
      --model k2-fsa/OmniVoice \
      --workspace /kaggle/working/OmniVoiceStudio \
      --asr-model openai/whisper-small.en \
      --asr-device cpu \
      --share


## Local workspace reminder

The stable hostname solves **network identity**, not persistence. The actual projects, voices, `jobs.json`, `section-status.json`, and `project-queue.json` still live under `/kaggle/working/OmniVoiceStudio` and disappear when the Kaggle session is discarded. Persistent storage remains a separate later layer.
